In [1]:
!pip install pandas numpy matplotlib seaborn scipy statsmodels scikit-learn

  Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.8.0-cp312-cp312-win_amd64.whl (8.0 MB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   ------------- -------------------------- 1/3 [joblib]
   -------------------------- ------------- 2/3 [scikit-learn]
   -------------------------- -------

In [2]:
# Attempt to import required libraries and provide graceful fallbacks/install hints
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import statsmodels.api as sm
    import statsmodels.formula.api as smf
    from statsmodels.stats.multicomp import pairwise_tukeyhsd
    from statsmodels.stats.outliers_influence import variance_inflation_factor
    from sklearn.model_selection import train_test_split
    import warnings

    warnings.filterwarnings('ignore')
    sns.set_theme(style="whitegrid", palette="muted")

except ModuleNotFoundError as e:
    missing = str(e)
    print(f"A required library is missing: {missing}")
    print(
        "If running locally, please install missing packages. For example, to install statsmodels:\n"
        "  pip install statsmodels\n"
        "To install scikit-learn if needed:\n"
        "  pip install scikit-learn\n"
        "If you're in a restricted environment without internet access, consider removing or guarding code that depends on the missing package."
    )
    # Minimal fallback to allow notebook to continue without failing import cell
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    import scipy.stats as stats
    import warnings
    warnings.filterwarnings('ignore')
    try:
        sns.set_theme(style="whitegrid", palette="muted")
    except Exception:
        pass

# Note for downstream cells:
# - If statsmodels wasn't available, variables like `sm`, `smf`, `pairwise_tukeyhsd`, `variance_inflation_factor` won't be defined.
# - Before using them, add runtime checks or install the dependency as suggested above.

> This cell installs any missing Python packages required later in the notebook\. It is safe to re\-run and produces minimal output\. If packages are installed, please re\-run the import cell above to ensure fresh imports in this session\.

## Phase 1: Data Acquisition and Preprocessing
We begin by understanding our data shape, locating missing values, handling extreme outliers, and creating categorical proxy variables.

In [3]:
# 1. Load Data
data = pd.read_csv("data/raw/data.csv")
print(f"Raw Data Shape: {data.shape}\n")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/data.csv'

In [ ]:
# View the first 5 rows
print(data.head())

# View the column names
print(data.columns)

   StudyHours  Attendance  Resources  Extracurricular  Motivation  Internet  \
0          19          64          1                0           0         1   
1          19          64          1                0           0         1   
2          19          64          1                0           0         1   
3          19          64          1                1           0         1   
4          19          64          1                1           0         1   

   Gender  Age  LearningStyle  OnlineCourses  Discussions  \
0       0   19              2              8            1   
1       0   23              3             16            0   
2       0   28              1             19            0   
3       0   19              2              8            1   
4       0   23              3             16            0   

   AssignmentCompletion  ExamScore  EduTech  StressLevel  FinalGrade  
0                    59         40        0            1           3  
1               

In [ ]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14003 entries, 0 to 14002
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   StudyHours            14003 non-null  int64
 1   Attendance            14003 non-null  int64
 2   Resources             14003 non-null  int64
 3   Extracurricular       14003 non-null  int64
 4   Motivation            14003 non-null  int64
 5   Internet              14003 non-null  int64
 6   Gender                14003 non-null  int64
 7   Age                   14003 non-null  int64
 8   LearningStyle         14003 non-null  int64
 9   OnlineCourses         14003 non-null  int64
 10  Discussions           14003 non-null  int64
 11  AssignmentCompletion  14003 non-null  int64
 12  ExamScore             14003 non-null  int64
 13  EduTech               14003 non-null  int64
 14  StressLevel           14003 non-null  int64
 15  FinalGrade            14003 non-null  int64
dtypes: i

In [ ]:
print(data.describe())

         StudyHours    Attendance     Resources  Extracurricular  \
count  14003.000000  14003.000000  14003.000000     14003.000000   
mean      19.987431     80.194316      1.104406         0.594158   
std        5.890637     11.472181      0.697362         0.491072   
min        5.000000     60.000000      0.000000         0.000000   
25%       16.000000     70.000000      1.000000         0.000000   
50%       20.000000     80.000000      1.000000         1.000000   
75%       24.000000     90.000000      2.000000         1.000000   
max       44.000000    100.000000      2.000000         1.000000   

         Motivation      Internet        Gender           Age  LearningStyle  \
count  14003.000000  14003.000000  14003.000000  14003.000000   14003.000000   
mean       0.905806      0.925516      0.551953     23.532172       1.515461   
std        0.695896      0.262566      0.497311      3.514293       1.112941   
min        0.000000      0.000000      0.000000     18.000000      

In [ ]:
columns_to_keep = ['Age', 'Gender', 'StudyHours', 'Attendance', 'AssignmentCompletion', 'ExamScore']
df = data[columns_to_keep].copy()

In [ ]:
# 2. Destroy Duplicates
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Dropped {initial_rows - len(df)} duplicate rows.")

Dropped 9803 duplicate rows.


In [ ]:
# 3. Handle Missing Values
print("\nMissing values per column before cleaning:")
print(df.isnull().sum())
df = df.dropna()
print(f"Total rows remaining after dropping NaNs: {len(df)}")


Missing values per column before cleaning:
Age                     0
Gender                  0
StudyHours              0
Attendance              0
AssignmentCompletion    0
ExamScore               0
dtype: int64
Total rows remaining after dropping NaNs: 4200


In [ ]:
# 4. Feature Engineering (The Buckets for Slide 5)
def categorize_completion(val):
    if val >= 80:
        return "High"
    elif val >= 50:
        return "Moderate"
    else:
        return "Low"

df['SelfAssessment_Level'] = df['AssignmentCompletion'].apply(categorize_completion)
# Lock the order so "Low" is always compared against "High" in charts
df['SelfAssessment_Level'] = pd.Categorical(df['SelfAssessment_Level'], categories=["Low", "Moderate", "High"], ordered=True)

In [ ]:
# 5. Outlier Removal using Tukey's IQR on ExamScore
Q1 = df['ExamScore'].quantile(0.25)
Q3 = df['ExamScore'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

df_clean = df[(df['ExamScore'] >= lower_bound) & (df['ExamScore'] <= upper_bound)]
print(f"\nDropped {len(df) - len(df_clean)} extreme outliers from ExamScore.")


Dropped 0 extreme outliers from ExamScore.


In [ ]:
# 6. Save the gold-standard dataset
df_clean.to_csv("data/processed/cleaned_data.csv", index=False)
print("Data cleaning complete. Clean file saved as 'cleaned_data.csv'.")

Data cleaning complete. Clean file saved as 'cleaned_data.csv'.


In [ ]:
# 3. Outlier Detection (Tukey's IQR Method on ExamScore)
Q1 = df['ExamScore'].quantile(0.25)
Q3 = df['ExamScore'].quantile(0.75)
IQR = Q3 - Q1
lower_bound, upper_bound = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

outliers = df[(df['ExamScore'] < lower_bound) | (df['ExamScore'] > upper_bound)]
print(f"\nDetected {len(outliers)} outliers in ExamScore. Removing them for robust analysis.")
df = df[(df['ExamScore'] >= lower_bound) & (df['ExamScore'] <= upper_bound)].copy()

# 4. Feature Engineering: Self-Assessment Categorization
df['SelfAssessment_Level'] = pd.cut(
    df['AssignmentCompletion'],
    bins=[-np.inf, 49, 79, np.inf],
    labels=['Low', 'Moderate', 'High'],
    ordered=True
)
print("\nCleaned Data Sample:")
display(df.head())

NameError: name 'data' is not defined